# Kuroshio R6.5 — 训练期跨分支信息丢弃（Exchange Dropout）机制资格实验

以 R6.1-CrossPhys / R6.4-GN8 为父架构，不改冻结流轴、EOF、LSTM 宽度与物理辅助损失。训练时随机遮蔽跨分支隐藏信息通道，测试时不遮蔽；探索候选 `p=0.35/0.60`。

**时间与公平性规则**：2014/2019 是已反复使用的研发年份，不是盲测。所有模型相同的训练/验证拆分、两个种子 71/72；与 R6.4 原有物理损失 LSTM、GN8 和 BN 控制对比。严禁把 QA 标签喂给训练；本 Notebook 只允许从 R6.4 的**真实完整预测结果**复用基线，不得改变真实分数。

**节约算力**：正常情况下直接读取本地 R6.4 结果包复用开发资格阶段对照，只训练两个新增 CrossDrop 模型；若对照归档缺失则明确要求提供原归档，而不擅自换成弱基线。

**自动结果交付**：每完成一年/种子即 ZIP + SHA256；成功、失败或未过 Gate 均提供 Jupyter 本机下载链接。

In [ ]:
# ===== 0. 配置：先核实三个目录；改好后从头顺序运行 =====
from dataclasses import dataclass, asdict
from pathlib import Path
import os, json, math, gc, time, random, hashlib, warnings, sys
import numpy as np
import pandas as pd

@dataclass
class CFG:
    ROOT: str = '/mnt/d/KuroshioDaily1993_2025'
    DATASET_REL: str = 'final/dataset/daily_dual_branch_paths_1993_2025_FINAL.npz'
    HANDOFF_REL: str = 'final/reports/LONG_DAILY_1993_2025_HANDOFF.json'
    QC_INDEX_REL: str = 'final/reports/最终逐日路径索引.csv'
    MANUAL_QUEUE_REL: str = 'reports/07_人工复核队列.csv'
    MANUAL_METADATA_REL: str = 'manual_qc/metadata'
    OUTPUT_REL: str = 'R6_5_跨分支交换随机屏蔽实验'
    HISTORY: int = 180
    LEADS: tuple = (30, 60, 90)
    TEST_YEARS: tuple = (2005, 2009, 2013, 2017, 2021, 2024)
    PURGE_DAYS: int = 7
    MAX_INPUT_GAP: int = 2
    MAX_MISSING_FRAC: float = 0.03
    MIN_DATA_YEARS: int = 25
    MIN_ANNUAL_COVERAGE: float = 0.85
    MIN_TRAIN: int = 500
    MIN_VAL: int = 100
    MIN_TEST: int = 100
    DROP_BRANCH1_FIRST: int = 12  # 43->31; same old representation
    BRANCH2_MAX_LON: float = 145.0
    EOF_VARIANCE_TARGET: float = 0.99
    BATCH_SIZE: int = 32
    EPOCHS_B1: int = 70
    EPOCHS_B2: int = 70
    PATIENCE: int = 12
    LR: float = 0.001
    WEIGHT_DECAY: float = 0.0001
    DROPOUT: float = 0.30
    SEEDS: tuple = (20,21,22,23,24)
    AR_LAGS: tuple = (3, 7)
    AR_ALPHAS: tuple = (0.1, 1., 10.)
    CLIM_HALF_WINDOW_DAYS: int = 15
    CLIM_MIN_SAMPLES: int = 10
    BOOT_N: int = 1000
    BOOT_BLOCK_DAYS: int = 30
    FORCE_TRAIN: bool = False
    USE_AMP: bool = True
    RUN_FULL_TRAINING: bool = False   # R5.3仅机制资格实验，严禁直接转长训
    TRAIN_OTHER_HYBRIDS: bool = False # 下一轮网络结构比较可启用，不改baseline主表
    FIG_DPI: int = 300
    PILOT_TEST_YEARS: tuple = (2009, 2017)  # QA: 2008/2016, inner early-stop: 2007/2015
    PILOT_SEEDS: tuple = (71, 72)
    PILOT_EPOCHS: int = 36
    PILOT_PATIENCE: int = 8
    GATE_MIN_GAIN_KM: float = 0.50
    GATE_MAX_LEAD_REGRESSION_KM: float = 2.0

    CONFIRM_PILOT_YEARS: tuple = (2013, 2021)  # QA 2012/2020; 曾经被研发流程涉及，不能称独立盲测
    MAX_PARAM_OVERHEAD_RATIO: float = 0.15
    REQUIRE_ALL_LEADS_NONINFERIOR: bool = True
    REQUIRE_BOTH_QA_YEARS_GAIN: bool = True
    ARCHIVE_INCLUDE_WEIGHTS: bool = False
    ARCHIVE_INCLUDE_CACHE: bool = False
    ARCHIVE_MAX_FILE_MB: int = 150
    RUN_CONFIRMATION: bool = True   # 仅阶段一通过才运行阶段二；不自动正式长训
    RAW_TRACK_DIR_REL: str = 'final/tracks_final'
    MAX_MISSING_DP_DAY_FRAC: float = 0.005
    MAX_DP_TEMPORAL_P95_RATIO: float = 1.40
    RAW_EVAL_ISSUES_PER_QA_YEAR: int = 36
    REBUILD_DP: bool = False
    DP_REPORT_EVERY: int = 250
    ARCHIVE_INCLUDE_WEIGHTS: bool = False
    ARCHIVE_INCLUDE_CACHE: bool = False

cfg=CFG(); ROOT=Path(cfg.ROOT); DS=ROOT/cfg.DATASET_REL; OUT=ROOT/cfg.OUTPUT_REL
DATA=OUT/'实验数据'; CKPT=OUT/'模型权重'; PRED=OUT/'逐样本预测'; REP=OUT/'报告'; FZH=OUT/'图表'/'中文'; FEN=OUT/'图表'/'英文'
for d in (DATA,CKPT,PRED,REP,FZH,FEN): d.mkdir(parents=True,exist_ok=True)
print(json.dumps(asdict(cfg),indent=2,ensure_ascii=False))
ACTIVE_REP='Original'
RAW_TRACK_DIR=ROOT/cfg.RAW_TRACK_DIR_REL
if not RAW_TRACK_DIR.is_dir(): raise FileNotFoundError(f'找不到原始完整流轴目录: {RAW_TRACK_DIR}')
DERIVED=OUT/'派生表示_不覆盖冻结数据';DERIVED.mkdir(exist_ok=True,parents=True)
ARCHIVE_PATH=ROOT/'R61_上传给ChatGPT_物理距离损失资格实验.zip'
ARCHIVE_SHA=Path(str(ARCHIVE_PATH)+'.sha256')
if not DS.exists(): raise FileNotFoundError(f'未发现真正的冻结数据：{DS}。请先运行人工QC notebook最后的冻结与数据构建单元。')
# R5.7 不启用 DP，不改变冻结目标。训练合同使用资产内容 SHA256。
ACTIVE_REP='Original'
A2_ORIGINAL=None
sign=hashlib.sha256(DS.read_bytes()).hexdigest()
PARENT_R57=ROOT/'R5_7_原表示混合模型稳定性资格实验'/'模型权重'
print('R6.5 Original frozen: causal training, no QA target leakage')


In [ ]:
# ===== 1. GPU与字体/依赖检查（不合格立即停止，避免白训练）=====
import importlib.util
required=('torch','scipy','sklearn','matplotlib','tqdm','pandas','numpy')
missing=[v for v in required if importlib.util.find_spec(v) is None]
if missing: raise RuntimeError('WSL环境缺依赖：'+', '.join(missing)+'；用当前Kernel对应python执行 python -m pip install torch scipy scikit-learn matplotlib tqdm pandas numpy')
import torch, torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from scipy.spatial import cKDTree
from sklearn.linear_model import Ridge
import matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties
from tqdm.auto import tqdm
print('kernel=',sys.executable,'torch=',torch.__version__,'CUDA=',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('本轮RTX5080正式训练要求CUDA GPU；请检查WSL nvidia-smi 与 PyTorch CUDA构建。')
print('GPU=',torch.cuda.get_device_name(0)); DEV=torch.device('cuda')
torch.set_float32_matmul_precision('high')
font_paths=['/mnt/c/Windows/Fonts/msyh.ttc','/mnt/c/Windows/Fonts/simhei.ttf','/mnt/c/Windows/Fonts/simsun.ttc','/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc']
CJK_NAME=None
for path in font_paths:
    if Path(path).exists():
        try:
            font_manager.fontManager.addfont(path)
            CJK_NAME=FontProperties(fname=path).get_name(); break
        except Exception: pass
if CJK_NAME is None:
    raise RuntimeError('中文字体缺失：请在WSL安装 sudo apt install fonts-noto-cjk 或检查 Windows /mnt/c/Windows/Fonts，避免绘图乱码。')
print('Chinese font=',CJK_NAME)


In [ ]:
# ===== 2. 最终数据读取、年度覆盖、人工QC冻结核对 =====
with np.load(DS,allow_pickle=False) as z:
    dates=pd.DatetimeIndex(pd.to_datetime(z['dates'])).normalize()
    b1=np.asarray(z['branch1_lon'],np.float32)
    lat1=np.asarray(z['branch1_target_lats'],float)
    b2=np.asarray(z['branch2_lat'],np.float32)
    lon2=np.asarray(z['branch2_target_lons'],float)
    source=np.asarray(z['source']).astype(str) if 'source' in z.files else np.array(['unknown']*len(dates))
if not (len(dates)==len(b1)==len(b2)==len(source)): raise RuntimeError('dates / 两分支 / source 数量不一致')
if dates.has_duplicates: raise RuntimeError('发现重复日期，禁止将其直接用于训练')
ord=np.argsort(dates.values); dates=dates[ord]; b1=b1[ord];b2=b2[ord];source=source[ord]
if not np.isfinite(b1).all() or not np.isfinite(b2).all(): raise RuntimeError('冻结数据中含 NaN/Inf')

# 物理范围异常只作质控警示（不随意删除或修改路径），便于发现断面插值外推造成的不可信日
suspicious=(~((b1>=119)&(b1<=140)).all(axis=1)) | (~((b2>=15)&(b2<=50)).all(axis=1))
if suspicious.any():
    suspect_df=pd.DataFrame({'异常日期':dates[suspicious], '原因':'双分支坐标超出宽松物理范围，需人工核对'})
    suspect_df.to_csv(DATA/'01_待人工复查的物理范围异常路径.csv',index=False,encoding='utf-8-sig')
    warnings.warn(f'发现 {suspicious.sum()} 天路径越过宽松物理边界，建议正式训练前核实此文件')

if b1.shape[1]<=cfg.DROP_BRANCH1_FIRST: raise RuntimeError('分支1维数过小')
b1=b1[:,cfg.DROP_BRANCH1_FIRST:];lat1=lat1[cfg.DROP_BRANCH1_FIRST:]
keep2=lon2<=cfg.BRANCH2_MAX_LON+1e-8;b2=b2[:,keep2];lon2=lon2[keep2]
if len(b1)!=len(b2) or len(lat1)!=b1.shape[1] or len(lon2)!=b2.shape[1]: raise RuntimeError('双分支物理坐标不匹配')
if dates.min().year>1993 or dates.max().year<2024: raise RuntimeError('数据不覆盖1993–2024所需区间，不可宣称多年全历史验证')
cal=pd.date_range(dates.min(),dates.max(),freq='D'); pos=cal.get_indexer(dates)
valid=np.zeros(len(cal),bool);valid[pos]=True
A1=np.full((len(cal),b1.shape[1]),np.nan,np.float32);A2=np.full((len(cal),b2.shape[1]),np.nan,np.float32)
A1[pos]=b1; A2[pos]=b2
# causal forward fill (ONLY history, never target). All targets later read A1/A2 raw by exact date.
def causal_fill(raw,limit): return pd.DataFrame(raw).ffill(limit=limit).to_numpy(dtype=np.float32)
F1=causal_fill(A1,cfg.MAX_INPUT_GAP);F2=causal_fill(A2,cfg.MAX_INPUT_GAP)
annual=pd.DataFrame({'日期':cal,'有效':valid});annual['年份']=annual['日期'].dt.year
cover=annual.groupby('年份',as_index=False).agg(理论天数=('日期','size'),有效天数=('有效','sum'))
cover['有效比例']=cover['有效天数']/cover['理论天数'];cover.to_csv(DATA/'01_逐年覆盖率.csv',index=False,encoding='utf-8-sig')
if (cover['有效比例']>=cfg.MIN_ANNUAL_COVERAGE).sum()<cfg.MIN_DATA_YEARS:
    raise RuntimeError('具备≥85%有效覆盖的年份不足25年，请优先排查未完成人工修复或缺失年；详情见01_逐年覆盖率.csv')
hand=ROOT/cfg.HANDOFF_REL
if hand.exists():
    ho=json.loads(hand.read_text(encoding='utf-8'))
    print('HANDOFF final_daily_paths=',ho.get('final_daily_paths'))
    if int(ho.get('final_daily_paths',-1))!=len(dates): raise RuntimeError('HANDOFF与冻结npz样本数不一致')
else: warnings.warn('未发现HANDOFF.json；冻结数据可供资格检查，但交接记录尚未核实')
idxp=ROOT/cfg.QC_INDEX_REL
if idxp.exists():
    idx=pd.read_csv(idxp)
    if 'status' in idx.columns:
        unresolved=idx[~idx['status'].isin(['final','missing_raw','unusable'])]
        if len(unresolved): raise RuntimeError(f'最终逐日索引仍有 {len(unresolved)} 条未处理状态，停止训练')
queue=ROOT/cfg.MANUAL_QUEUE_REL; mdir=ROOT/cfg.MANUAL_METADATA_REL
if queue.exists():
    qq=pd.read_csv(queue)
    qdates=pd.to_datetime(qq['date']).dt.strftime('%Y%m%d') if len(qq) else []
    missing_meta=[q for q in qdates if not (mdir/f'{q}.json').exists()]
    if missing_meta: raise RuntimeError(f'{len(missing_meta)} 个manual复核队列日期尚无处理metadata: {missing_meta[:10]}')
    bad_meta=[]
    allowed={'manual_corrected','accepted_auto_after_manual_qc','unusable'}
    for q in qdates:
        try:
            state=json.loads((mdir/f'{q}.json').read_text(encoding='utf-8')).get('status')
            if state not in allowed:bad_meta.append((q,state))
        except Exception as e:bad_meta.append((q,repr(e)))
    if bad_meta:raise RuntimeError(f'{len(bad_meta)} 条人工QC仍不是已确认状态: {bad_meta[:10]}')
# CMEMS来源切换边界连续性检查：仅警告，不把真实大蛇行直接当成数据损坏
edge=pd.Timestamp('2025-01-01')
prev=np.flatnonzero(dates<edge)
post=np.flatnonzero(dates>=edge)
if len(prev) and len(post):
    i,j=int(prev[-1]),int(post[0])
    if (dates[j]-dates[i]).days<=5:
        d1=float(np.sqrt(np.mean((b1[j]-b1[i])**2)))
        d2=float(np.sqrt(np.mean((b2[j]-b2[i])**2)))
        pd.DataFrame([{'产品交界前':dates[i],'交界后':dates[j],
                       '分支1逐日经度RMSE_度':d1,'分支2逐日纬度RMSE_度':d2}]).to_csv(
            DATA/'01_2024到2025产品交界连续性.csv',index=False,encoding='utf-8-sig')
        if d1>1.0 or d2>1.5:
            warnings.warn('2024-2025 CMEMS来源交界发生较大路径跳变，应先检查跨产品物理一致性与流轴QC')
print('DATA PASS:',len(dates),'days;',dates.min().date(),'to',dates.max().date(),'; B1',b1.shape[1],'; B2',b2.shape[1])
display(cover)


In [ ]:
# ===== 3. 创建样本索引，不展开 [样本,180,特征] 巨型数组 =====
# target index = issue+lead; test targets require true valid day. input missing <=2 continuous days.
L=max(cfg.LEADS); H=cfg.HISTORY
missing=(~valid).astype(np.int32)
issue_indices=[];audit=[]
for i in range(H-1,len(cal)-L):
    if not valid[i]: continue
    win=missing[i-H+1:i+1]
    if win.mean()>cfg.MAX_MISSING_FRAC: continue
    if len(win) and np.max(np.diff(np.flatnonzero(np.r_[1,win==0,1]))-1)>cfg.MAX_INPUT_GAP: continue
    if any(not valid[i+d] for d in cfg.LEADS): continue
    if not (np.isfinite(F1[i-H+1:i+1]).all() and np.isfinite(F2[i-H+1:i+1]).all()): continue
    issue_indices.append(i)
ISS=np.asarray(issue_indices,np.int32)
if len(ISS)==0: raise RuntimeError('没有有效180→30/60/90 anchors，需检查缺日与冻结数据')
anchor=pd.DataFrame({'起报日期':cal[ISS]})
for l in cfg.LEADS: anchor[f'{l}天目标日期']=cal[ISS+l]
anchor.to_csv(DATA/'02_全部有效预测起报日期.csv',index=False,encoding='utf-8-sig')
print('anchor count=',len(ISS),' first=',cal[ISS[0]],'last=',cal[ISS[-1]])


In [ ]:
# ===== 4. 真正的 expanding-window 多年后报：test year = 2005/2009/2013/2017/2021/2024 =====
FOLDS=[];fr=[]; splitrows=[]
for fold_id,year in enumerate(cfg.TEST_YEARS,1):
    valstart=pd.Timestamp(year-1,1,1); teststart=pd.Timestamp(year,1,1)
    testend=pd.Timestamp(year,12,31)
    train_cut=valstart-pd.Timedelta(days=cfg.PURGE_DAYS)
    val_cut=teststart-pd.Timedelta(days=cfg.PURGE_DAYS)
    issues=cal[ISS]; latest=cal[ISS+L]
    tr=ISS[(latest<train_cut)]
    va=ISS[(issues>=valstart)&(issues<teststart)&(latest<val_cut)]
    te=ISS[(issues>=teststart)&(issues<=testend)]
    if len(tr)<cfg.MIN_TRAIN or len(va)<cfg.MIN_VAL or len(te)<cfg.MIN_TEST:
        raise RuntimeError(f'Fold {year} 样本不足: tr={len(tr)} val={len(va)} test={len(te)}；请查该年的日路径覆盖和连续缺日')
    if not(cal[tr+L].max()<cal[va].min()-pd.Timedelta(days=cfg.PURGE_DAYS) and cal[va+L].max()<cal[te].min()-pd.Timedelta(days=cfg.PURGE_DAYS)):
        raise RuntimeError(f'Fold {year} 发现严格时序泄漏')
    if (len(set(cal[tr+L])&set(cal[va+L])) or len(set(cal[tr+L])&set(cal[te+L])) or len(set(cal[va+L])&set(cal[te+L]))):
        raise RuntimeError(f'Fold {year} label date overlap')
    ff={'fold':fold_id,'year':year,'train':tr,'val':va,'test':te,'train_cut':train_cut,'val_cut':val_cut}
    FOLDS.append(ff)
    fr.append({'折次':fold_id,'测试年份':year,'训练样本':len(tr),'验证样本':len(va),'测试样本':len(te),
               '训练最晚标签日期':cal[tr+L].max(),'验证最早起报日期':cal[va].min(),
               '验证最晚标签日期':cal[va+L].max(),'测试最早起报日期':cal[te].min(),'测试最晚标签日期':cal[te+L].max(),'因果审计':'PASS'})
    for label,ix in [('训练',tr),('验证',va),('测试',te)]:
        splitrows.extend({'折次':fold_id,'测试年份':year,'集合':label,'起报日期':str(cal[i].date()),'最晚目标日期':str(cal[i+L].date())} for i in ix)
foldtab=pd.DataFrame(fr); foldtab.to_csv(DATA/'03_多年滚动后报折次及防泄漏审计.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(splitrows).to_csv(DATA/'03_逐样本时序划分.csv',index=False,encoding='utf-8-sig')
print('[STRICT MULTI-YEAR CAUSAL HINDCAST] ALL PASS'); display(foldtab)


In [ ]:
# ===== 6. Original-only：不生成DP，不修改冻结数据 =====
A2_ORIGINAL=A2.copy()
F2_ORIGINAL=F2.copy()
print('ORIGINAL FROZEN REPRESENTATION ONLY; dataset SHA256:',sign)


In [ ]:
# ===== 5. 每折 train-only 标准化 + 自适应 99% EOF =====
# 使用该fold训练期的独立日记录，而不是重叠180d窗口重复叠加做SVD；内存恒定在 O(feature²)
class DailyEOF:
    def fit(self,obs,target=0.99):
        obs=np.asarray(obs,np.float64)
        self.mu=np.mean(obs,axis=0); self.sd=np.std(obs,axis=0); self.sd[self.sd<1e-6]=1.
        z=(obs-self.mu)/self.sd
        cov=(z.T@z)/max(1,len(z)-1)
        eig,vec=np.linalg.eigh(cov)
        idx=np.argsort(eig)[::-1];eig=np.maximum(eig[idx],0);vec=vec[:,idx]
        rat=np.cumsum(eig)/(eig.sum()+1e-12); n=int(np.searchsorted(rat,target)+1)
        self.V=np.asarray(vec[:,:n],np.float32);self.explained=float(rat[n-1]);self.n=n
        return self
    def pc(self,x): return (((np.asarray(x)-self.mu)/self.sd)@self.V).astype(np.float32)
    def invert(self,pc): return ((np.asarray(pc)@self.V.T)*self.sd+self.mu).astype(np.float32)
    def z(self,x): return ((np.asarray(x)-self.mu)/self.sd).astype(np.float32)

def make_fold_data(f):
    train_days=(cal< f['train_cut']) & valid
    pp1=DailyEOF().fit(A1[train_days],cfg.EOF_VARIANCE_TARGET)
    pp2=DailyEOF().fit(A2[train_days],cfg.EOF_VARIANCE_TARGET)
    for pp in (pp1,pp2):
        if pp.explained<cfg.EOF_VARIANCE_TARGET-1e-6: raise RuntimeError('EOF 99%训练方差Gate未通过')
    # approx memory: ~17k days * 2*(31+277) floats; never expands 180 day windows
    z1=pp1.z(F1);z2=pp2.z(F2)
    pcs1=pp1.pc(F1);pcs2=pp2.pc(F2)
    ppath=CKPT/ACTIVE_REP/f'第{f["fold"]}折_{f["year"]}年'; ppath.mkdir(exist_ok=True,parents=True)
    np.savez_compressed(ppath/'训练期预处理参数.npz',
        b1_mu=pp1.mu,b1_sd=pp1.sd,b1_V=pp1.V,b2_mu=pp2.mu,b2_sd=pp2.sd,b2_V=pp2.V,
        branch1_lat=lat1,branch2_lon=lon2,train_label_cutoff=str(f['train_cut'].date()))
    return {'pp1':pp1,'pp2':pp2,'z1':z1,'z2':z2,'p1':pcs1,'p2':pcs2,'path':ppath}

# 只预审一折，防止全局重复保留大数组
_testFD=make_fold_data(FOLDS[0]);print('fold1 EOF modes:',_testFD['pp1'].n,_testFD['pp2'].n,'explained:',_testFD['pp1'].explained,_testFD['pp2'].explained)
del _testFD;gc.collect()
print('[TRAIN-ONLY EOF 99%] PASS')


In [ ]:
# ===== 6. 动态加载数据集，训练时每个batch只展开当前180天窗口 =====
class CausalWindowDataset(Dataset):
    def __init__(self,issues,scaled,pcs):
        self.issues=np.asarray(issues,np.int32);self.scaled=scaled;self.pcs=pcs
    def __len__(self): return len(self.issues)
    def __getitem__(self,j):
        i=int(self.issues[j]);z=self.scaled[i-H+1:i+1]
        dz=np.empty_like(z);dz[0]=0;dz[1:]=z[1:]-z[:-1]
        x=np.concatenate([z,dz],axis=-1)
        y=np.stack([self.pcs[i+d] for d in cfg.LEADS])
        return torch.from_numpy(x.copy()),torch.from_numpy(y.copy())

class StepEncoder(nn.Module):
    # 与旧原稿的per-step TCN/Conv编码保持一致：输入路径特征卷积，然后LSTM承担时间建模
    def __init__(self,feature,branch,kind='tcn'):
        super().__init__();self.kind=kind;self.drop=nn.Dropout(cfg.DROPOUT)
        if kind=='tcn':
            channels=16 if branch==1 else 32;kernel=3 if branch==1 else 5
            self.conv=nn.Conv1d(1,channels,kernel,dilation=2)
            self.bn=nn.BatchNorm1d(channels)
            conv_len=feature+2-2*(kernel-1)
            self.fc=nn.Linear(channels*conv_len,32 if branch==1 else 64)
            self.out_dim=self.fc.out_features
        elif kind=='conv':
            c1=16 if branch==1 else 32;c2=32 if branch==1 else 64
            self.c1=nn.Conv1d(1,c1,3,padding=1);self.c2=nn.Conv1d(c1,c2,3,padding=1)
            self.out_dim=c2*feature
        else: raise ValueError(kind)
    def forward(self,x):
        B,T,F=x.shape; z=x.reshape(B*T,1,F)
        if self.kind=='tcn':
            z=torch.nn.functional.pad(z,(2,0)) # 与R3原架构一致的左侧填充
            z=self.drop(torch.relu(self.bn(self.conv(z))))
            z=self.drop(torch.relu(self.fc(z.flatten(1))))
        else:
            z=self.drop(torch.relu(self.c1(z)));z=self.drop(torch.relu(self.c2(z)));z=z.flatten(1)
        return z.reshape(B,T,-1)

class Hybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,kind='tcn',recur='lstm'):
        super().__init__();self.enc=StepEncoder(feature,branch,kind)
        h=32 if branch==1 else 64
        R=nn.GRU if recur=='gru' else nn.LSTM
        self.rnn=R(self.enc.out_dim,h,batch_first=True)
        self.fc=nn.Linear(h,3*pc_dim);self.pc_dim=pc_dim
    def forward(self,x):
        h,_=self.rnn(self.enc(x));return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

class MatchedLSTM(nn.Module):
    def __init__(self,feature,pc_dim,hidden):
        super().__init__();self.rnn=nn.LSTM(feature,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim);self.pc_dim=pc_dim
    def forward(self,x):
        h,_=self.rnn(x);return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

def nparams(m): return sum(p.numel() for p in m.parameters())
def match_lstm(feature,pc_dim,branch):
    target=nparams(Hybrid(feature,pc_dim,branch))
    # Pytorch single-layer LSTM: 4h(feature+h+2) + 3pc*h + 3pc
    opts=range(8,1025)
    h=min(opts,key=lambda hh:abs(4*hh*(feature+hh+2)+(3*pc_dim)*(hh+1)-target))
    actual=4*h*(feature+h+2)+(3*pc_dim)*(h+1)
    if abs(actual-target)/target>0.08: raise RuntimeError('Matched-LSTM参数量相对差>8%，不满足公平基线')
    return h,target,actual

def make_net(name,branch,fd):
    feature=fd[f'z{branch}'].shape[1]*2;pdim=fd[f'pp{branch}'].n
    if name=='Matched-LSTM':
        h,_,_=match_lstm(feature,pdim,branch);return MatchedLSTM(feature,pdim,h)
    if name=='EOF-TCN-LSTM': return Hybrid(feature,pdim,branch,'tcn','lstm')
    if name=='EOF-Conv-LSTM': return Hybrid(feature,pdim,branch,'conv','lstm')
    if name=='EOF-Conv-GRU': return Hybrid(feature,pdim,branch,'conv','gru')
    if name=='EOF-TCN-GRU': return Hybrid(feature,pdim,branch,'tcn','gru')
    raise ValueError(name)




# ===== R5.3：路径坐标/一阶差分分为两个空间输入通道 =====
class ChannelSeparatedEncoder(nn.Module):
    """单时刻空间卷积；输入[B,T,2*M] => Conv1d [B*T,2,M]."""
    def __init__(self,feature,branch):
        super().__init__()
        assert feature%2==0, '需要[x, delta]等长拼接输入'
        self.raw_dim=feature//2
        base_c=16 if branch==1 else 32
        k=3 if branch==1 else 5
        channels=base_c*2
        # 正确的对称空间填充：dilation=2 -> padding=k-1，输出仍为M
        self.conv=nn.Conv1d(2,channels,kernel_size=k,dilation=2,padding=k-1)
        self.bn=nn.BatchNorm1d(channels)
        self.fc=nn.Linear(channels*self.raw_dim,32 if branch==1 else 64)
        self.out_dim=self.fc.out_features
        self.drop=nn.Dropout(cfg.DROPOUT)
    def channelize(self,x):
        B,T,F=x.shape
        if F!=self.raw_dim*2: raise RuntimeError('R53特征维不匹配')
        return torch.stack((x[:,:,:self.raw_dim],x[:,:,self.raw_dim:]),dim=2).reshape(B*T,2,self.raw_dim)
    def forward(self,x):
        B,T,F=x.shape
        z=self.channelize(x)
        z=self.drop(torch.relu(self.bn(self.conv(z))))
        if z.shape[2]!=self.raw_dim:raise RuntimeError('R53 spatial-conv输出宽度发生变化')
        z=self.drop(torch.relu(self.fc(z.flatten(1))))
        return z.reshape(B,T,-1)

class ChannelHybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,projection=None,use_pc_skip=False):
        super().__init__()
        self.enc=ChannelSeparatedEncoder(feature,branch)
        self.use_pc_skip=bool(use_pc_skip)
        self.raw_dim=feature//2
        self.pc_dim=pc_dim
        if self.use_pc_skip:
            self.register_buffer('V',torch.as_tensor(projection,dtype=torch.float32))
        hidden=32 if branch==1 else 64
        in_dim=self.enc.out_dim+(2*pc_dim if self.use_pc_skip else 0)
        self.rnn=nn.LSTM(in_dim,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim)
    def eof_skip(self,x):
        if not self.use_pc_skip: raise RuntimeError('no PC skip in this model')
        return torch.cat((x[:,:,:self.raw_dim]@self.V,x[:,:,self.raw_dim:]@self.V),dim=-1)
    def forward(self,x):
        z=self.enc(x)
        if self.use_pc_skip: z=torch.cat((z,self.eof_skip(x)),dim=-1)
        h,_=self.rnn(z)
        return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

R53_CANDIDATES=('R53-ChannelConv','R53-ChannelConv-PCConcat')
R53_BASELINES=('Matched-LSTM','EOF-TCN-LSTM','R53-CapacityMatchedLSTM')
R53_ALL=R53_BASELINES+R53_CANDIDATES

def make_r53_model(name,branch,fd):
    feature=fd[f'z{branch}'].shape[1]*2
    pdim=fd[f'pp{branch}'].n
    if name in ('Matched-LSTM','EOF-TCN-LSTM'):
        return make_net(name,branch,fd)
    if name=='R53-CapacityMatchedLSTM':
        ref=ChannelHybrid(feature,pdim,branch,fd[f'pp{branch}'].V,use_pc_skip=True)
        h,_,_=matched_hidden_for_target(feature,pdim,nparams(ref))
        return MatchedLSTM(feature,pdim,h)
    if name=='R53-ChannelConv':
        return ChannelHybrid(feature,pdim,branch)
    if name=='R53-ChannelConv-PCConcat':
        return ChannelHybrid(feature,pdim,branch,fd[f'pp{branch}'].V,use_pc_skip=True)
    raise ValueError(name)

def matched_hidden_for_target(feature,pdim,target):
    # 单层 PyTorch LSTM 参数解析式，替代遍历建立 1017 个网络实例
    def count(h):return 4*h*(feature+h+2)+3*pdim*(h+1)
    best=min(range(8,1025),key=lambda h:abs(count(h)-target))
    actual=count(best)
    ratio=abs(actual-target)/max(target,1)
    if ratio>0.08:raise RuntimeError(f'参数匹配误差过大：{ratio:.2%}')
    return best,actual,ratio

def model_loss(pred,target,fd,branch,name):
    # 与R5.2完全相同的训练目标：只用PC-MSE做结构归因。
    return torch.nn.functional.mse_loss(pred,target)


In [ ]:
# ===== 8. 物理公里误差：局地投影+最近点相邻线段的双向正交路径距离 =====
# 对于仅122-145E的分支2，重建得到共同定义域；不是对每一点简单经纬度平方误差。
# 下方是双向point-to-polyline近似cross-track，使用KDTree近邻三顶点及其邻接segments。
def merge_path(branch1,branch2):
    a=np.asarray(branch1,float);b=np.asarray(branch2,float)
    candidates=np.flatnonzero(lon2>a[-1]);
    k=int(candidates[np.argmin((lon2[candidates]-a[-1])**2+(b[candidates]-lat1[-1])**2)]) if len(candidates) else len(lon2)
    xs=np.r_[a,lon2[k:]];ys=np.r_[lat1,b[k:]]
    good=np.isfinite(xs)&np.isfinite(ys)
    return xs[good],ys[good]

def one_direction(A,B):
    # A,B: 2d coordinates (km); approximate nearest polyline segment for each A point
    if len(A)<2 or len(B)<2: return np.array([np.nan])
    tree=cKDTree(B);_,knn=tree.query(A,k=min(3,len(B)))
    knn=np.asarray(knn);knn=knn[:,None] if knn.ndim==1 else knn
    ix=np.unique(np.clip(np.concatenate([knn.ravel()-1,knn.ravel()]),0,len(B)-2))
    # segment shortlist globally may grow, but polynomial per path length (~300) and is vectorized
    v=B[ix+1]-B[ix];w=A[:,None,:]-B[ix][None,:,:]
    den=np.maximum(np.sum(v*v,axis=1),1e-12)
    t=np.clip(np.sum(w*v[None,:,:],axis=2)/den[None,:],0,1)
    q=B[ix][None,:,:]+t[:,:,None]*v[None,:,:]
    d=np.sqrt(np.sum((A[:,None,:]-q)**2,axis=2))
    return d.min(axis=1)

def errors_for_pair(pred1,pred2,truth1,truth2):
    x,y=merge_path(pred1,pred2);xt,yt=merge_path(truth1,truth2)
    lat0=np.mean(np.r_[y,yt]);lon0=np.mean(np.r_[x,xt]);c=np.cos(np.deg2rad(lat0))
    A=np.column_stack([(x-lon0)*111.32*c,(y-lat0)*111.32]);B=np.column_stack([(xt-lon0)*111.32*c,(yt-lat0)*111.32])
    d=np.r_[one_direction(A,B),one_direction(B,A)]
    return float(np.mean(d**2)),float(np.mean(d)),float(np.percentile(d,90))

def record_errors(fold,model,seed,issues,p1,p2):
    out=[]
    for n,i in enumerate(issues):
        for k,lead in enumerate(cfg.LEADS):
            y1=A1[i+lead];y2=A2[i+lead]
            mse,mean,p90=errors_for_pair(p1[n,k],p2[n,k],y1,y2)
            out.append({'折次':fold['fold'],'测试年份':fold['year'],'模型':model,'种子':seed,'起报日期':str(cal[i].date()),'目标日期':str(cal[i+lead].date()),'提前期_天':int(lead),
                        '横向路径误差MSE_km2':mse,'横向路径平均误差_km':mean,'横向路径P90_km':p90,
                        '分支1经度RMSE_度':float(np.sqrt(np.mean((p1[n,k]-y1)**2))),
                        '分支2纬度RMSE_度':float(np.sqrt(np.mean((p2[n,k]-y2)**2)))})
    return pd.DataFrame(out)


In [ ]:
# ===== 9. R5 训练封装、物理评价、可恢复 checkpoints =====
from contextlib import nullcontext

def r5_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def run_predict(net, issues, fd, branch):
    ds=CausalWindowDataset(issues,fd[f'z{branch}'],fd[f'p{branch}'])
    ld=DataLoader(ds,batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0)
    arr=[];net.eval()
    with torch.no_grad():
        for xx,_ in ld: arr.append(net(xx.to(DEV)).float().cpu().numpy())
    return np.concatenate(arr,axis=0)

def fit_r5_branch(split, fd, name, seed, branch, root_dir, max_epochs=None, patience=None):
    max_epochs=cfg.PILOT_EPOCHS if max_epochs is None else max_epochs
    patience=cfg.PILOT_PATIENCE if patience is None else patience
    folder=root_dir/ACTIVE_REP/f'QA_{split["year"]-1}'/name/f'seed_{seed}'
    folder.mkdir(parents=True,exist_ok=True)
    weight=folder/f'branch{branch}_best.pt'; historyfile=folder/f'branch{branch}_history.csv'
    latest=folder/f'branch{branch}_latest.pt'
    # best checkpoint 附带实验合同签名；仅同一fold/样本划分/网络版本可复用
    contract={'name':name,'seed':seed,'branch':branch,'tr_n':int(len(split['train'])),
              'val_n':int(len(split['val'])),'tr_last':int(split['train'][-1]),
              'val_first':int(split['val'][0]),'val_last':int(split['val'][-1]),'train_first':int(split['train'][0]),'max_epochs':max_epochs,
              'feature_dims':[fd['z1'].shape[1],fd['z2'].shape[1]],'version':'R5.7-original-stability-v1','rep':ACTIVE_REP,'dataset_signature':sign}
    contract_id=hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()
    r5_seed(seed)
    net=make_r53_model(name,branch,fd).to(DEV)
    if weight.exists() and historyfile.exists() and not cfg.FORCE_TRAIN:
        ck=torch.load(weight,map_location='cpu',weights_only=True)
        if ck.get('contract_id')==contract_id:
            net.load_state_dict(ck['state']);return net,pd.read_csv(historyfile)
    ds_tr=CausalWindowDataset(split['train'],fd[f'z{branch}'],fd[f'p{branch}'])
    ds_va=CausalWindowDataset(split['val'],fd[f'z{branch}'],fd[f'p{branch}'])
    ld_tr=DataLoader(ds_tr,batch_size=cfg.BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True)
    ld_va=DataLoader(ds_va,batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
    opt=torch.optim.AdamW(net.parameters(),lr=cfg.LR,weight_decay=cfg.WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,factor=0.5,patience=4)
    amp=bool(cfg.USE_AMP)
    scaler=torch.amp.GradScaler('cuda',enabled=amp)
    start_epoch=1;best=float('inf');bestep=0;bad=0;hist=[]
    if latest.exists() and not cfg.FORCE_TRAIN:
        try:
            ck=torch.load(latest,map_location='cpu',weights_only=True)
            if ck.get('contract_id')==contract_id:
                net.load_state_dict(ck['state']);opt.load_state_dict(ck['opt']);scheduler.load_state_dict(ck['scheduler']);scaler.load_state_dict(ck['scaler'])
                start_epoch=int(ck['epoch'])+1;best=float(ck['best']);bestep=int(ck['bestep']);bad=int(ck['bad']);hist=ck['hist']
                print('[RESUME]',folder,'epoch',start_epoch)
        except Exception as e:print('[resume invalid]',e)
    for ep in range(start_epoch,max_epochs+1):
        net.train();ts=tn=0
        for x,y in ld_tr:
            x=x.to(DEV,non_blocking=True);y=y.to(DEV,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',enabled=amp):
                pred=net(x)
                loss=model_loss(pred.float(),y.float(),fd,branch,name)
            if not torch.isfinite(loss):raise RuntimeError(f'{name} B{branch}: non-finite loss')
            scaler.scale(loss).backward();scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(),1.0)
            scaler.step(opt);scaler.update();ts+=float(loss.detach())*len(x);tn+=len(x)
        net.eval();vs=vn=0
        with torch.no_grad():
            for x,y in ld_va:
                x=x.to(DEV);y=y.to(DEV)
                with torch.autocast('cuda',enabled=amp):
                    p=net(x);loss=model_loss(p.float(),y.float(),fd,branch,name)
                vs+=float(loss.detach())*len(x);vn+=len(x)
        val=vs/max(1,vn);scheduler.step(val)
        hist.append({'epoch':ep,'train_loss':ts/max(1,tn),'val_loss':val,'lr':opt.param_groups[0]['lr']})
        if val<best-1e-7:
            best=val;bestep=ep;bad=0
            torch.save({'state':{k:v.detach().cpu() for k,v in net.state_dict().items()},'contract_id':contract_id,'epoch':ep,'best_val':best},weight)
        else:bad+=1
        if ep%5==0 or bad>=patience or ep==max_epochs:
            torch.save({'state':{k:v.detach().cpu() for k,v in net.state_dict().items()},'opt':opt.state_dict(),
                        'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'epoch':ep,'best':best,
                        'bestep':bestep,'bad':bad,'hist':hist,'contract_id':contract_id},latest)
        if ep==1 or ep%10==0 or bad>=patience:print(f'{name} B{branch} seed{seed} ep{ep} train={hist[-1]["train_loss"]:.5f} val={val:.5f}')
        if bad>=patience:break
    pd.DataFrame(hist).to_csv(historyfile,index=False,encoding='utf-8-sig')
    if latest.exists():latest.unlink()
    ck=torch.load(weight,map_location='cpu',weights_only=True)
    net.load_state_dict(ck['state']);net.eval()
    return net,pd.DataFrame(hist)

def evaluate_issues(f,name,seed,fd,issues,bmodel1,bmodel2):
    return record_errors(f,name,seed,issues,bmodel1,bmodel2)

def get_predictions(name,split,fd,seed,folder,max_epochs=None,patience=None):
    branches=[]
    for branch in (1,2):
        net,hist=fit_r5_branch(split,fd,name,seed,branch,folder,max_epochs,patience)
        pr=run_predict(net,split.get('qa',split.get('test')),fd,branch)
        branches.append(fd[f'pp{branch}'].invert(pr))
        del net,pr;torch.cuda.empty_cache()
    return branches


In [ ]:
# ===== 7. 预先固定 QA 拆分：标签与早停严格先于 QA 日期 =====
PILOT_QA_YEARS = (2006,2014)
CONFIRM_QA_YEARS = (2011,2019)
R57_MODELS = ('Matched-LSTM','R53-CapacityMatchedLSTM','EOF-TCN-LSTM','R53-ChannelConv-PCConcat')
R57_SEEDS=tuple(cfg.PILOT_SEEDS)

# QA 年份不再依赖 R56 TEST_YEARS 白名单；其余合同完全一致。
def r57_pilot_split(qa_year):
    qa_start=pd.Timestamp(qa_year,1,1)
    next_start=pd.Timestamp(qa_year+1,1,1)
    inner_start=pd.Timestamp(qa_year-1,1,1)
    purge=pd.Timedelta(days=cfg.PURGE_DAYS)
    train=ISS[cal[ISS+L] < inner_start-purge]
    val=ISS[(cal[ISS]>=inner_start) & (cal[ISS+L]<qa_start-purge)]
    qa=ISS[(cal[ISS]>=qa_start) & (cal[ISS+L]<next_start-purge)]
    if min(len(train),len(val),len(qa))<min(cfg.MIN_TRAIN,cfg.MIN_VAL):
        raise RuntimeError(f'QA {qa_year} 样本量不足 train={len(train)} val={len(val)} qa={len(qa)}')
    if not (len(train)>=cfg.MIN_TRAIN and len(val)>=cfg.MIN_VAL and len(qa)>=cfg.MIN_VAL):
        raise RuntimeError(f'QA {qa_year} 数据资格未满足')
    assert cal[train+L].max()<cal[val].min()-purge
    assert cal[val+L].max()<cal[qa].min()-purge
    return {'fold':qa_year,'year':qa_year+1,'train':train,'val':val,'qa':qa,'train_cut':inner_start-purge}

PILOTS=[r57_pilot_split(y) for y in PILOT_QA_YEARS]
CONFIRMS=[r57_pilot_split(y) for y in CONFIRM_QA_YEARS]
SPLIT_ROWS=[]
for phase,fs in [('资格探索',PILOTS),('后续年份确认',CONFIRMS)]:
    for f in fs:
        SPLIT_ROWS.append({'阶段':phase,'QA年份':f['year']-1,'训练数':len(f['train']),'早停数':len(f['val']),'QA数':len(f['qa']),
          '训练最晚标签日期':str(cal[f['train']+L].max().date()),'早停最晚标签日期':str(cal[f['val']+L].max().date()),
          'QA最早起报日期':str(cal[f['qa']].min().date())})
split_df=pd.DataFrame(SPLIT_ROWS)
split_df.to_csv(DATA/'R60_01_严格因果时序拆分.csv',index=False,encoding='utf-8-sig')
display(split_df)
print('[R60 STRICT CAUSAL SPLIT] PASS')


## 阶段 1 · 隐状态耦合的最小机制

对同一输入历史分别进行 `ChannelSeparatedEncoder → EOF bypass → LSTM`，在两个 LSTM 的**最后隐状态**计算完毕后执行一次双向信息交换：

\[
 h_1^{\prime}=h_1+W_{2\to1}\tanh(h_2), \qquad
 h_2^{\prime}=h_2+W_{1\to2}\tanh(h_1).
\]

两个线性映射零初始化，不引入后处理纠偏或相位标签；其梯度从两个预测损失反向传播进入两条 LSTM 历史编码路径。

`R60-SelfHidden` 使用同分支映射作为容量对照；`R60-CrossHidden` 使用另一分支的隐状态。输出都是原有 EOF-PC 三提前期预测。

In [ ]:
# ===== R60. 隐状态的双向耦合，训练期端到端反向传播 =====
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from IPython.display import display, FileLink, Markdown
from contextlib import nullcontext

R60_QA_YEARS=tuple(PILOT_QA_YEARS)
R60_CONFIRM_YEARS=tuple(CONFIRM_QA_YEARS)
R60_SEEDS=tuple(cfg.PILOT_SEEDS)
R60_BASELINES=('Matched-LSTM','R53-CapacityMatchedLSTM','R53-ChannelConv-PCConcat')
R60_CANDIDATES=('R60-SelfHidden','R60-CrossHidden')
R60_MODELS=R60_BASELINES+R60_CANDIDATES
R60_EPOCHS=int(cfg.PILOT_EPOCHS)
R60_PATIENCE=int(cfg.PILOT_PATIENCE)

class JointPairDataset(Dataset):
    def __init__(self,issues,fd):
        self.issues=np.asarray(issues,dtype=np.int64)
        self.z1=fd['z1'];self.z2=fd['z2'];self.p1=fd['p1'];self.p2=fd['p2']
    def __len__(self): return len(self.issues)
    def __getitem__(self,j):
        i=int(self.issues[j])
        def pack(z):
            v=z[i-H+1:i+1]
            d=np.empty_like(v);d[0]=0;d[1:]=v[1:]-v[:-1]
            return torch.from_numpy(np.concatenate([v,d],axis=-1).copy())
        yy1=np.stack([self.p1[i+lead] for lead in cfg.LEADS]);yy2=np.stack([self.p2[i+lead] for lead in cfg.LEADS])
        return pack(self.z1),pack(self.z2),torch.from_numpy(yy1.copy()),torch.from_numpy(yy2.copy())

class JointHiddenEOF(nn.Module):
    def __init__(self,fd,mode='cross'):
        super().__init__()
        if mode not in ('cross','self'): raise ValueError(mode)
        self.mode=mode
        self.net1=make_r53_model('R53-ChannelConv-PCConcat',1,fd)
        self.net2=make_r53_model('R53-ChannelConv-PCConcat',2,fd)
        d1=int(self.net1.fc.in_features);d2=int(self.net2.fc.in_features)
        self.exchange1=nn.Linear(d2 if mode=='cross' else d1,d1)
        self.exchange2=nn.Linear(d1 if mode=='cross' else d2,d2)
        nn.init.zeros_(self.exchange1.weight);nn.init.zeros_(self.exchange1.bias)
        nn.init.zeros_(self.exchange2.weight);nn.init.zeros_(self.exchange2.bias)
    def _hidden(self,net,x):
        a=net.enc(x)
        a=torch.cat([a,net.eof_skip(x)],dim=-1)
        h,_=net.rnn(a)
        return h[:,-1]
    def forward(self,x1,x2):
        h1=self._hidden(self.net1,x1)
        h2=self._hidden(self.net2,x2)
        if self.mode=='cross':
            t1=h1+self.exchange1(torch.tanh(h2))
            t2=h2+self.exchange2(torch.tanh(h1))
        else:
            t1=h1+self.exchange1(torch.tanh(h1))
            t2=h2+self.exchange2(torch.tanh(h2))
        o1=self.net1.fc(t1).reshape(len(x1),3,self.net1.pc_dim)
        o2=self.net2.fc(t2).reshape(len(x2),3,self.net2.pc_dim)
        return o1,o2

def r60_count_models(fd):
    a=JointHiddenEOF(fd,'cross');b=JointHiddenEOF(fd,'self')
    base=sum(nparams(make_r53_model('R53-ChannelConv-PCConcat',k,fd)) for k in (1,2))
    large=sum(nparams(make_r53_model('R53-CapacityMatchedLSTM',k,fd)) for k in (1,2))
    rows=[{'模型':'R53 independent','参数量':base}, {'模型':'容量匹配 LSTM','参数量':large},
          {'模型':'R60-SelfHidden','参数量':nparams(b)}, {'模型':'R60-CrossHidden','参数量':nparams(a)}]
    return pd.DataFrame(rows)

f0=make_fold_data(PILOTS[0]);param=r60_count_models(f0)
param['相对大LSTM参数差_%']=(param['参数量']/param.loc[param['模型']=='容量匹配 LSTM','参数量'].iloc[0]-1)*100
param.to_csv(DATA/'R60_02_参数量公平审计.csv',index=False,encoding='utf-8-sig')
display(param)
if (param[param['模型'].str.startswith('R60-')]['相对大LSTM参数差_%'].abs()>5).any():
    raise RuntimeError('R60候选与大LSTM参数偏差>5%，未通过公平性合同')

# 测试两个模式输入输出、真实反向传播、零初始化扰动=0、跨分支改变应对另一输出有梯度
r5_seed(20261014)
for mode in ('self','cross'):
    m=JointHiddenEOF(f0,mode).to(DEV)
    x1=torch.randn(2,5,f0['z1'].shape[1]*2,device=DEV)
    x2=torch.randn(2,5,f0['z2'].shape[1]*2,device=DEV)
    m.eval()
    with torch.no_grad():
        y1,y2=m(x1,x2)
        assert y1.shape==(2,3,f0['pp1'].n) and y2.shape==(2,3,f0['pp2'].n)
        assert torch.allclose(y1,m.net1(x1),atol=1e-6,rtol=1e-6)
        assert torch.allclose(y2,m.net2(x2),atol=1e-6,rtol=1e-6)
    m.train();u,v=m(x1,x2);loss=u.square().mean()+v.square().mean();loss.backward()
    assert m.exchange1.weight.grad is not None and torch.isfinite(m.exchange1.weight.grad).all()
    assert m.exchange2.weight.grad is not None and torch.isfinite(m.exchange2.weight.grad).all()
    print('[R60 SMOKE]',mode,'PASS', nparams(m))
    del m
if DEV.type=='cuda': torch.cuda.empty_cache()
del f0
print('[R60 MODEL CAUSALITY AND CAPACITY PREFLIGHT] PASS')


## R6.1 独立机制：物理距离辅助损失

相比直接对 EOF 系数拟合 MSE，这个损失把预测 PC 与目标 PC 之差投影回经纬度，再转成各固定经纬线采样点的公里误差；辅助项采用 Smooth-L1，仍保留一部分 EOF-MSE 避免几何过度平滑。**它是物理量近似监督，不等于评估阶段的双向线段横向距离。** 共同 QA 评价函数 `errors_for_pair` 保持不变。

In [ ]:
# ===== R61.1. 固定损失合同 + 物理回投测试 =====
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import traceback, zipfile, hashlib, os, json, gc, time
from IPython.display import display, FileLink, Markdown
from datetime import timezone, datetime

R61_ALPHA = 0.40                  # 物理项权重（预先固定；不依据QA调整）
R61_DISTANCE_SCALE_KM = 25.0    # 用于数值归一化（不是评价阈值）
R61_HUBER_BETA = 0.5
R61_SEEDS = tuple(cfg.PILOT_SEEDS)
R61_EPOCHS = int(cfg.PILOT_EPOCHS)
R61_PATIENCE = int(cfg.PILOT_PATIENCE)
R61_MODELS = ('Matched-LSTM','R53-CapacityMatchedLSTM','R53-ChannelConv-PCConcat',
              'R61-LSTMPhys','R61-HybridPhys','R61-CrossPhys')
R61_CANDIDATES = ('R61-HybridPhys','R61-CrossPhys')
R61_OUTPUT_ZIP = ROOT/'R61_上传给ChatGPT_物理距离损失资格实验.zip'
R61_SHA = Path(str(R61_OUTPUT_ZIP)+'.sha256')
R61_DATA_CSV = DATA/'R61_02_逐样本预测误差.csv'
R61_CONTRACT = {'research_version':'R64-loss-alignment-baseline-v1','dataset_sha256':sign,
                'alpha':R61_ALPHA,'km_norm':R61_DISTANCE_SCALE_KM,'beta':R61_HUBER_BETA,
                'epochs':R61_EPOCHS,'patience':R61_PATIENCE,'seed_list':R61_SEEDS,
                'qa_years':R60_QA_YEARS,'confirmation_years':R60_CONFIRM_YEARS,
                'test_data_are_not_blind':True,'physical_loss_applies_to_lstm':True,
                'eof_train_only':True,'expected_models':list(R61_MODELS)}
(REP/'R61_实验预注册合同.json').write_text(json.dumps(R61_CONTRACT,ensure_ascii=False,indent=2),encoding='utf-8')

def r61_geometry_tensors(fd, branch, device):
    pp=fd[f'pp{branch}']
    V=torch.as_tensor(pp.V,dtype=torch.float32,device=device)
    sd=torch.as_tensor(pp.sd,dtype=torch.float32,device=device)
    if branch==1:
        scale=torch.as_tensor(111.32*np.cos(np.deg2rad(lat1)),dtype=torch.float32,device=device)
    else:
        scale=torch.ones_like(sd)*111.32
    if len(sd)!=len(scale) or V.shape!=(len(sd),pp.n):
        raise RuntimeError(f'Branch {branch} geometry / EOF dimensions mismatch')
    return V,sd*scale

def r61_loss(pred,target,fd,branch,alpha=R61_ALPHA):
    if pred.shape != target.shape:
        raise RuntimeError(f'R61 predicted PCs {pred.shape} != target {target.shape}')
    pc_mse=F.mse_loss(pred.float(),target.float())
    V,sd_km=r61_geometry_tensors(fd,branch,pred.device)
    delta_degree=(pred.float()-target.float())@V.T
    delta_km=delta_degree*sd_km[None,None,:]
    norm=delta_km/R61_DISTANCE_SCALE_KM
    phy=F.smooth_l1_loss(norm,torch.zeros_like(norm),beta=R61_HUBER_BETA)
    return (1-alpha)*pc_mse+alpha*phy

fd0=make_fold_data(PILOTS[0])
for br in (1,2):
    p=torch.randn(2,3,fd0[f'pp{br}'].n,device=DEV,requires_grad=True)
    t=torch.randn_like(p)
    obj=r61_loss(p,t,fd0,br)
    obj.backward()
    assert torch.isfinite(obj) and torch.isfinite(p.grad).all()
    assert torch.allclose(r61_loss(p,t,fd0,br,alpha=0),F.mse_loss(p,t),rtol=1e-6)
    print('[R61 PHYSICAL LOSS SMOKE] Branch',br,'PASS',float(obj.detach().cpu()))
del fd0
if DEV.type=='cuda':torch.cuda.empty_cache()
print('[R61 LOSS+CAUSAL CONTRACT PREFLIGHT] ALL PASS')

In [ ]:
# ===== R61.3 独立分支端到端训练：原架构或容量匹配LSTM，同物理损失 =====
from torch.utils.data import DataLoader
from torch import nn
from contextlib import nullcontext
R61_WEIGHTS=CKPT/'R61物理训练'; R61_WEIGHTS.mkdir(parents=True,exist_ok=True)

def r61_train_independent(f,fd,seed,branch,model_type):
    assert model_type in ('R61-HybridPhys','R61-LSTMPhys')
    source_model='R53-ChannelConv-PCConcat' if model_type=='R61-HybridPhys' else 'R53-CapacityMatchedLSTM'
    r5_seed(seed)
    net=make_r53_model(source_model,branch,fd).to(DEV)
    tr=DataLoader(CausalWindowDataset(f['train'],fd[f'z{branch}'],fd[f'p{branch}']),batch_size=cfg.BATCH_SIZE,shuffle=True,num_workers=0)
    va=DataLoader(CausalWindowDataset(f['val'],fd[f'z{branch}'],fd[f'p{branch}']),batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0)
    directory=R61_WEIGHTS/f'{f["year"]-1}'/str(seed)/model_type/f'b{branch}'
    directory.mkdir(parents=True,exist_ok=True)
    latest=directory/'latest.pt';best_file=directory/'best.pt';hist_file=directory/'history.csv'
    contract={**R61_CONTRACT,'phase_year':f['year']-1,'seed':seed,'type':model_type,
              'branch':branch,'train_last':int(f['train'][-1]),'val_last':int(f['val'][-1]),
              'pcdim':fd[f'pp{branch}'].n}
    cid=hashlib.sha256(json.dumps(contract,ensure_ascii=False,sort_keys=True).encode()).hexdigest()
    if best_file.exists() and hist_file.exists() and not cfg.FORCE_TRAIN:
        bk=torch.load(best_file,map_location='cpu',weights_only=True)
        if bk['cid']==cid:
            net.load_state_dict(bk['state']);net.eval()
            return net
    opt=torch.optim.AdamW(net.parameters(),lr=cfg.LR,weight_decay=cfg.WEIGHT_DECAY)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,patience=4,factor=.5)
    amp=(DEV.type=='cuda' and bool(cfg.USE_AMP))
    scaler=torch.amp.GradScaler('cuda',enabled=amp)
    best=float('inf');bad=0;begin=1;log=[]
    if latest.exists() and not cfg.FORCE_TRAIN:
        try:
            ck=torch.load(latest,map_location='cpu',weights_only=True)
            if ck['cid']==cid:
                net.load_state_dict(ck['state']);opt.load_state_dict(ck['opt']);sch.load_state_dict(ck['sch']);scaler.load_state_dict(ck['scaler'])
                best=float(ck['best']);bad=int(ck['bad']);begin=int(ck['epoch'])+1;log=ck['log']
        except Exception as e:print('[R61 RESUME SKIP]',repr(e))
    for epoch in range(begin,R61_EPOCHS+1):
        net.train();total=0.;count=0
        for x,y in tr:
            x=x.to(DEV);y=y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',enabled=amp):
                out=net(x)
                loss=r61_loss(out.float(),y.float(),fd,branch)
            if not torch.isfinite(loss):raise RuntimeError('R61 nonfinite loss')
            scaler.scale(loss).backward();scaler.unscale_(opt);nn.utils.clip_grad_norm_(net.parameters(),1)
            scaler.step(opt);scaler.update()
            total+=float(loss.detach())*len(x);count+=len(x)
        net.eval();vtotal=0.;vcount=0
        with torch.no_grad():
            for x,y in va:
                x=x.to(DEV);y=y.to(DEV)
                with torch.autocast('cuda',enabled=amp):
                    loss=r61_loss(net(x).float(),y.float(),fd,branch)
                vtotal+=float(loss.detach())*len(x);vcount+=len(x)
        v=vtotal/max(1,vcount);sch.step(v)
        log.append({'epoch':epoch,'train_loss':total/max(1,count),'val_loss':v})
        if v<best-1e-7:
            best=v;bad=0;torch.save({'cid':cid,'state':{k:vv.cpu() for k,vv in net.state_dict().items()}},best_file)
        else:bad+=1
        if epoch==1 or epoch%5==0 or bad>=R61_PATIENCE:
            print('[R61 TRAIN]',model_type,'year',f['year']-1,'seed',seed,'B',branch,'ep',epoch,'val',round(v,5))
            torch.save({'cid':cid,'state':{k:vv.cpu() for k,vv in net.state_dict().items()},
                        'opt':opt.state_dict(),'sch':sch.state_dict(),'scaler':scaler.state_dict(),
                        'epoch':epoch,'best':best,'bad':bad,'log':log},latest)
        if bad>=R61_PATIENCE:break
    if not best_file.exists():raise RuntimeError('R61 no best checkpoint')
    pd.DataFrame(log).to_csv(hist_file,index=False,encoding='utf-8-sig')
    if latest.exists():latest.unlink()
    net.load_state_dict(torch.load(best_file,map_location='cpu',weights_only=True)['state']);net.eval()
    return net

def r61_predict_independent(f,fd,seed,name):
    pp=[]
    for branch in (1,2):
        net=r61_train_independent(f,fd,seed,branch,name)
        pc=run_predict(net,f['qa'],fd,branch)
        pp.append(fd[f'pp{branch}'].invert(pc))
        del net,pc
        if DEV.type=='cuda':torch.cuda.empty_cache()
    return pp

print('[R61 INDEPENDENT TRAINER READY]')

In [ ]:
# ===== R65.1 固定实验合同和 R6.4 真实基线复用 =====
from datetime import datetime, timezone
from IPython.display import display, FileLink
import zipfile, io, traceback, copy

R65_SEEDS=(71,72)
R65_EXPLORATORY_YEARS=(2014,2019)
R65_CONFIRM_YEARS=(2006,2011)
R65_MODELS=('R65-LSTMPhys','R65-CrossPhys-BN','R65-CrossPhys-GN8','R65-CrossDrop35','R65-CrossDrop60')
R65_CANDIDATES=('R65-CrossDrop35','R65-CrossDrop60')
R65_EPOCHS=36
R65_PATIENCE=8
R65_MIN_GAIN_KM=0.5
R65_ARCHIVE=ROOT/'R65_上传给ChatGPT_跨分支信息随机屏蔽资格实验.zip'
R65_SHA=Path(str(R65_ARCHIVE)+'.sha256')
R65_RESULT=DATA/'R65_逐样本真实预测误差.csv'
R65_WEIGHTS=CKPT/'R65_Checkpoints'
R65_WEIGHTS.mkdir(parents=True,exist_ok=True)
R65_FOLDS={y:next(f for f in PILOTS+CONFIRMS if f['year']-1==y) for y in R65_EXPLORATORY_YEARS+R65_CONFIRM_YEARS}
R65_PROTOCOL={
    'version':'R65_GN8_CrossExchangeDrop_v1',
    'dataset_sha256':sign,
    'data_representation':'Original frozen',
    'history_days':cfg.HISTORY,
    'lead_days':list(cfg.LEADS),
    'physical_loss_alpha':float(R61_ALPHA),
    'development_years':list(R65_EXPLORATORY_YEARS),
    'confirmation_development_years':list(R65_CONFIRM_YEARS),
    'seeds':list(R65_SEEDS),
    'epochs':R65_EPOCHS,'patience':R65_PATIENCE,
    'dropout':cfg.DROPOUT,
    'cross_exchange_dropout':[0.35,0.60],
    'lr':cfg.LR, 'weight_decay':cfg.WEIGHT_DECAY,
    'reference':'same-data same-split R6.4 genuine prediction archive',
    'model_choice_from_confirm_years':False,
    'all_QA_years_development_not_blind':True
}
(REP/'R65_00_预注册实验合同.json').write_text(
    json.dumps(R65_PROTOCOL,ensure_ascii=False,indent=2),encoding='utf-8')

# 在 R6.4 原路径下优先读取逐样本文件；或者读取 R6.4 自带 ZIP
R64_OUT=ROOT/'R6_4_归一化跨年稳健性实验'
R64_CSV=R64_OUT/'实验数据'/'R64_逐样本真实预测误差.csv'
R64_ZIP=ROOT/'R64_上传给ChatGPT_归一化稳健性资格实验.zip'

def load_r64_baseline():
    if R64_CSV.is_file():
        source='csv'; raw=pd.read_csv(R64_CSV)
        proto_path=R64_OUT/'报告'/'R64_00_预先固定实验合同.json'
        if not proto_path.exists():
            raise FileNotFoundError(f'缺少原 R6.4 数据合同：{proto_path}')
        proto=json.loads(proto_path.read_text(encoding='utf-8'))
    elif R64_ZIP.is_file():
        source='zip'
        with zipfile.ZipFile(R64_ZIP) as arc:
            if arc.testzip(): raise RuntimeError('原 R6.4 ZIP CRC 未通过')
            raw=pd.read_csv(io.BytesIO(arc.read('实验数据/R64_逐样本真实预测误差.csv')))
            proto=json.loads(arc.read('报告/R64_00_预先固定实验合同.json'))
    else:
        raise FileNotFoundError('没有找到 R6.4 原始逐样本结果。请把 R64_上传给ChatGPT_归一化稳健性资格实验.zip 放到项目根目录；不要重训基线。')
    if proto['dataset_sha256'] != sign:
        raise RuntimeError('R6.4 数据SHA256与当前冻结数据不同，禁止混用旧结果')
    required={
        'R64-LSTMPhys':'R65-LSTMPhys',
        'R64-CrossPhys-BN':'R65-CrossPhys-BN',
        'R64-CrossPhys-GN8':'R65-CrossPhys-GN8'}
    df=raw[(raw['阶段']=='快速资格')&raw['QA年份'].isin(R65_EXPLORATORY_YEARS)
          &raw['种子'].isin(R65_SEEDS)&raw['模型'].isin(required)].copy()
    df['模型']=df['模型'].map(required)
    df['阶段']='快速资格'
    expected={(y,s,m,l) for y in R65_EXPLORATORY_YEARS for s in R65_SEEDS
             for m in required.values() for l in cfg.LEADS}
    actual=set(zip(df['QA年份'].astype(int),df['种子'].astype(int),df['模型'],df['提前期_天'].astype(int)))
    if actual != expected: raise RuntimeError(f'R6.4 对照不完整；缺少组合：{list(expected-actual)[:5]}')
    if not df['横向路径误差MSE_km2'].apply(np.isfinite).all():
        raise RuntimeError('R6.4 对照包含非有限误差')
    for (y,s,m,l),g in df.groupby(['QA年份','种子','模型','提前期_天']):
        expected_dates={str(cal[int(i)].date()) for i in R65_FOLDS[int(y)]['qa']}
        if set(g['起报日期'].astype(str))!=expected_dates or len(g)!=len(expected_dates):
            raise RuntimeError(f'旧 R6.4 对照起报日期不一致：{y,s,m,l}')
    if R65_RESULT.exists() and not cfg.FORCE_TRAIN:
        previous=pd.read_csv(R65_RESULT)
        previous=previous[~previous['模型'].isin(required.values())]
        df=pd.concat([previous,df],ignore_index=True)
    df.to_csv(R65_RESULT,index=False,encoding='utf-8-sig')
    print(f'[R65 REUSE R64 REAL BASELINES] source={source}, rows={len(df)}; dataset contract PASS')

load_r64_baseline()

In [ ]:
# ===== R65.2 跨分支交换随机屏蔽：GN8，零初始化，原始主干不变 =====
import torch.nn.functional as F

class CrossExchangeDropGN(JointHiddenEOF):
    def __init__(self,fd,drop_p):
        super().__init__(fd,'cross')
        self.drop_p=float(drop_p)
        assert 0 <= self.drop_p < 1.0
        for net in (self.net1,self.net2):
            ch=net.enc.conv.out_channels
            assert ch%8==0
            net.enc.bn=nn.GroupNorm(8,ch)

    def forward(self,x1,x2):
        h1=self._hidden(self.net1,x1)
        h2=self._hidden(self.net2,x2)
        m1=self.exchange1(torch.tanh(h2))
        m2=self.exchange2(torch.tanh(h1))
        # 训练期对隐藏特征做独立 dropout；推理时恒等，严禁随机测试增强
        m1=F.dropout(m1,p=self.drop_p,training=self.training)
        m2=F.dropout(m2,p=self.drop_p,training=self.training)
        y1=self.net1.fc(h1+m1).reshape(len(x1),3,self.net1.pc_dim)
        y2=self.net2.fc(h2+m2).reshape(len(x2),3,self.net2.pc_dim)
        return y1,y2

def r65_model(fd,name):
    if name=='R65-CrossDrop35':return CrossExchangeDropGN(fd,0.35)
    if name=='R65-CrossDrop60':return CrossExchangeDropGN(fd,0.60)
    if name=='R65-CrossPhys-GN8':return CrossExchangeDropGN(fd,0.0)
    if name=='R65-CrossPhys-BN':return JointHiddenEOF(fd,'cross')
    raise ValueError(name)

f0=make_fold_data(R65_FOLDS[2014])
try:
    control=r65_model(f0,'R65-CrossPhys-GN8').to(DEV)
    models=[(name,r65_model(f0,name).to(DEV)) for name in ('R65-CrossDrop35','R65-CrossDrop60')]
    ref=sum(nparams(make_r53_model('R53-CapacityMatchedLSTM',branch,f0)) for branch in (1,2))
    stats=[{'模型':'R65-LSTMPhys','参数量':ref}]
    stats.extend([{'模型':name,'参数量':nparams(net)} for name,net in [('R65-CrossPhys-GN8',control)]+models])
    stats=pd.DataFrame(stats)
    stats['相对LSTM参数差_%']=100*(stats['参数量']/ref-1)
    stats.to_csv(DATA/'R65_01_参数量公平审计.csv',index=False,encoding='utf-8-sig')
    display(stats)
    assert (stats['相对LSTM参数差_%'].abs()<=5).all()
    assert all(nparams(net)==nparams(control) for _,net in models)
    x1=torch.randn(2,6,2*f0['z1'].shape[1],device=DEV)
    x2=torch.randn(2,6,2*f0['z2'].shape[1],device=DEV)
    for name,net in models:
        net.train()
        a,b=net(x1,x2)
        assert a.shape==(2,3,f0['pp1'].n) and b.shape==(2,3,f0['pp2'].n)
        (a.square().mean()+b.square().mean()).backward()
        assert all(torch.isfinite(p.grad).all() for p in net.parameters() if p.grad is not None)
        net.eval()
        with torch.no_grad():
            u1,u2=net(x1,x2);v1,v2=net(x1,x2)
        assert torch.equal(u1,v1) and torch.equal(u2,v2)
        print('[R65 MODEL PREFLIGHT]',name,'PASS')
    print('[R65 ORIGINAL LSTM, GN8 PARAMETER CONTRACT] PASS')
finally:
    del f0,control,models,x1,x2
    gc.collect()
    if DEV.type=='cuda': torch.cuda.empty_cache()

In [ ]:
# ===== R65.3 有合同签名的训练、早停、预测与断点恢复 =====
from torch.utils.data import DataLoader


def r65_contract(f,fd,seed,name):
    c={**R65_PROTOCOL,'QA_year':int(f['year']-1),'seed':int(seed),'name':name,
       'tr_end':int(f['train'][-1]),'va_end':int(f['val'][-1]),
       'nPC1':int(fd['pp1'].n),'nPC2':int(fd['pp2'].n)}
    return hashlib.sha256(json.dumps(c,ensure_ascii=False,sort_keys=True).encode()).hexdigest()


def r65_train(f,fd,seed,name):
    if name not in R65_CANDIDATES and name!='R65-CrossPhys-GN8':raise ValueError(name)
    r5_seed(seed)
    m=r65_model(fd,name).to(DEV)
    folder=R65_WEIGHTS/str(f['year']-1)/str(seed)/name
    folder.mkdir(parents=True,exist_ok=True)
    bestfile=folder/'best.pt';lastfile=folder/'latest.pt';history=folder/'history.csv'
    cid=r65_contract(f,fd,seed,name)
    if bestfile.exists() and not cfg.FORCE_TRAIN:
        ck=torch.load(bestfile,map_location='cpu',weights_only=True)
        if ck['cid']==cid:
            m.load_state_dict(ck['state']);m.eval()
            print('[R65 REUSE CHECKPOINT]',f['year']-1,seed,name)
            return m
    tr=DataLoader(JointPairDataset(f['train'],fd),batch_size=cfg.BATCH_SIZE,shuffle=True,num_workers=0)
    va=DataLoader(JointPairDataset(f['val'],fd),batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0)
    opt=torch.optim.AdamW(m.parameters(),lr=cfg.LR,weight_decay=cfg.WEIGHT_DECAY)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,patience=4,factor=.5)
    amp=DEV.type=='cuda' and bool(cfg.USE_AMP)
    scaler=torch.amp.GradScaler('cuda',enabled=amp)
    start=1;best=float('inf');bad=0;hist=[]
    if lastfile.exists() and not cfg.FORCE_TRAIN:
        try:
            ck=torch.load(lastfile,map_location='cpu',weights_only=True)
            if ck['cid']==cid:
                m.load_state_dict(ck['state']);opt.load_state_dict(ck['opt']);sch.load_state_dict(ck['sch'])
                scaler.load_state_dict(ck['scaler']);start=int(ck['epoch'])+1
                best=float(ck['best']);bad=int(ck['bad']);hist=ck['history']
                print('[R65 RESUME]',name,f['year']-1,seed,'start',start)
        except Exception as e: print('[R65 RESUME IGNORED]',repr(e))
    if bad>=R65_PATIENCE: start=R65_EPOCHS+1
    for epoch in range(start,R65_EPOCHS+1):
        m.train();trsum=0.;trn=0
        for x1,x2,y1,y2 in tr:
            x1,x2,y1,y2=[t.to(DEV) for t in (x1,x2,y1,y2)]
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',enabled=amp):
                a,b=m(x1,x2)
                loss=.5*(r61_loss(a.float(),y1.float(),fd,1)+r61_loss(b.float(),y2.float(),fd,2))
            if not torch.isfinite(loss):raise RuntimeError('R65 train loss is nonfinite')
            scaler.scale(loss).backward();scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(m.parameters(),1.0)
            scaler.step(opt);scaler.update()
            trsum+=float(loss.detach())*len(x1);trn+=len(x1)
        m.eval();vasum=0.;van=0
        with torch.no_grad():
            for x1,x2,y1,y2 in va:
                x1,x2,y1,y2=[t.to(DEV) for t in (x1,x2,y1,y2)]
                with torch.autocast('cuda',enabled=amp):
                    a,b=m(x1,x2)
                    loss=.5*(r61_loss(a.float(),y1.float(),fd,1)+r61_loss(b.float(),y2.float(),fd,2))
                vasum+=float(loss.detach())*len(x1);van+=len(x1)
        val=vasum/max(1,van);sch.step(val)
        if val < best-1.e-7:
            best=val;bad=0
            torch.save({'cid':cid,'state':{k:v.detach().cpu() for k,v in m.state_dict().items()},
                        'epoch':epoch,'best_val':best},bestfile)
        else: bad+=1
        hist.append({'epoch':int(epoch),'train_loss':trsum/max(1,trn),
                     'val_loss':float(val),'best_loss':float(best),
                     'lr':float(opt.param_groups[0]['lr'])})
        pd.DataFrame(hist).to_csv(history,index=False,encoding='utf-8-sig')
        torch.save({'cid':cid,'state':{k:v.detach().cpu() for k,v in m.state_dict().items()},
                    'opt':opt.state_dict(),'sch':sch.state_dict(),'scaler':scaler.state_dict(),
                    'epoch':epoch,'bad':bad,'best':best,'history':hist},lastfile)
        if epoch==1 or epoch%5==0 or bad>=R65_PATIENCE:
            print('[R65 TRAIN]',f['year']-1,seed,name,'ep',epoch,'train',round(trsum/trn,5),'val',round(val,5),'bad',bad)
        if bad>=R65_PATIENCE:break
    if not bestfile.exists():raise RuntimeError(f'R65 no best checkpoint {folder}')
    ck=torch.load(bestfile,map_location='cpu',weights_only=True)
    if ck['cid']!=cid:raise RuntimeError('R65 best checkpoint mismatch')
    m.load_state_dict(ck['state']);m.eval()
    return m


def r65_predict_pair(model,fd,issues):
    ld=DataLoader(JointPairDataset(issues,fd),batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0)
    all1=[];all2=[];model.eval()
    with torch.no_grad():
        for x1,x2,*_ in ld:
            p1,p2=model(x1.to(DEV),x2.to(DEV))
            all1.append(p1.float().cpu().numpy());all2.append(p2.float().cpu().numpy())
    return (fd['pp1'].invert(np.concatenate(all1)),fd['pp2'].invert(np.concatenate(all2)))


def r65_record(f,name,seed,p1,p2,phase):
    rows=[]
    for j,issue in enumerate(f['qa']):
        for k,lead in enumerate(cfg.LEADS):
            mse,mean,p90=errors_for_pair(p1[j,k],p2[j,k],A1[issue+lead],A2[issue+lead])
            rows.append({'阶段':phase,'QA年份':int(f['year']-1),'模型':name,'种子':int(seed),
                '起报日期':str(cal[issue].date()),'目标日期':str(cal[issue+lead].date()),
                '提前期_天':int(lead),'横向路径误差MSE_km2':float(mse),
                '平均距离_km':float(mean),'P90_km':float(p90)})
    return pd.DataFrame(rows)


def r65_eval(f,fd,seed,name,phase):
    if R65_RESULT.exists() and not cfg.FORCE_TRAIN:
        old=pd.read_csv(R65_RESULT)
        rows=old[(old['阶段']==phase)&(old['QA年份']==f['year']-1)&
                 (old['种子']==seed)&(old['模型']==name)]
        n_expected=len(f['qa'])*len(cfg.LEADS)
        if len(rows)==n_expected and rows['起报日期'].nunique()==len(f['qa']):
            print('[R65 REUSE PRED]',name,'year',f['year']-1,'seed',seed)
            return
    if name=='R65-LSTMPhys':
        p1,p2=r61_predict_independent(f,fd,seed,'R61-LSTMPhys')
    else:
        m=r65_train(f,fd,seed,name)
        p1,p2=r65_predict_pair(m,fd,f['qa'])
        del m
    rows=r65_record(f,name,seed,p1,p2,phase)
    old=pd.read_csv(R65_RESULT) if R65_RESULT.exists() else pd.DataFrame(columns=rows.columns)
    old=old[~((old['阶段']==phase)&(old['QA年份']==f['year']-1)&
              (old['种子']==seed)&(old['模型']==name))]
    pd.concat([old,rows],ignore_index=True).to_csv(R65_RESULT,index=False,encoding='utf-8-sig')
    print('[R65 EVAL DONE]',name,f['year']-1,seed,len(rows))
    if DEV.type=='cuda':torch.cuda.empty_cache()

In [ ]:
# ===== R65.4 先自动归档，再评估，不允许失败阻断结果下载 =====
from IPython.display import display,FileLink
import matplotlib.pyplot as plt


def r65_archive(stage='in_progress',show_link=True):
    (REP/'R65_打包状态.json').write_text(json.dumps({
        'status':stage,'time':datetime.now(timezone.utc).isoformat(),
        'QA_not_blind':True},ensure_ascii=False,indent=2),encoding='utf-8')
    files=[p for p in OUT.rglob('*') if p.is_file() and p.suffix.lower() in
           {'.csv','.json','.md','.log','.txt','.pdf','.png','.svg'} and
           '模型权重' not in p.parts and p.stat().st_size<=40*1024**2]
    files=sorted(files)
    tmp=Path(str(R65_ARCHIVE)+'.partial');manifest=[]
    with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for p in files:
            b=p.read_bytes();rel=p.relative_to(OUT).as_posix()
            z.writestr(rel,b)
            manifest.append({'path':rel,'sha256':hashlib.sha256(b).hexdigest(),'bytes':len(b)})
        z.writestr('R65_归档清单.json',json.dumps(manifest,ensure_ascii=False,indent=2))
    with zipfile.ZipFile(tmp) as z:
        bad=z.testzip()
        if bad:raise RuntimeError('R65 CRC failed '+bad)
    tmp.replace(R65_ARCHIVE)
    hasher=hashlib.sha256()
    with R65_ARCHIVE.open('rb') as fp:
        for ch in iter(lambda:fp.read(1024*1024),b''):hasher.update(ch)
    R65_SHA.write_text(f'{hasher.hexdigest()}  {R65_ARCHIVE.name}\n',encoding='utf-8')
    print('[R65 ZIP]',stage,'files',len(files),R65_ARCHIVE)
    if show_link:display(FileLink(str(R65_ARCHIVE)))


def r65_audit(stage,years):
    raw=pd.read_csv(R65_RESULT)
    raw=raw[(raw['阶段']==stage)&raw['QA年份'].isin(years)]
    expected={(y,s,m,l) for y in years for s in R65_SEEDS for m in R65_MODELS for l in cfg.LEADS}
    actual=set(zip(raw['QA年份'].astype(int),raw['种子'].astype(int),raw['模型'],raw['提前期_天'].astype(int)))
    if expected!=actual:raise RuntimeError(f'R65 incomplete; missing={list(expected-actual)[:8]}')
    for (y,s,m,l),g in raw.groupby(['QA年份','种子','模型','提前期_天']):
        days={str(cal[int(i)].date()) for i in R65_FOLDS[int(y)]['qa']}
        if len(g)!=len(days) or set(g['起报日期'].astype(str))!=days:
            raise RuntimeError(f'R65 date mismatch: {y,s,m,l}')
        if not np.isfinite(g['横向路径误差MSE_km2']).all():raise RuntimeError('nonfinite error')
    group=raw.groupby(['QA年份','种子','模型','提前期_天'],as_index=False).agg(
        RMSE_km=('横向路径误差MSE_km2',lambda x:float(np.sqrt(x.mean()))))
    group.to_csv(DATA/f'R65_{stage}_年份种子提前期误差.csv',index=False,encoding='utf-8-sig')
    agg=group.groupby(['模型','提前期_天'],as_index=False)['RMSE_km'].mean()
    agg.to_csv(DATA/f'R65_{stage}_模型平均误差.csv',index=False,encoding='utf-8-sig')
    table=group.pivot(index=['QA年份','种子','提前期_天'],columns='模型',values='RMSE_km')
    decisions=[]
    for name in R65_CANDIDATES:
        g=table['R65-LSTMPhys']-table[name]
        gn=table['R65-CrossPhys-GN8']-table[name]
        years_gain=g.groupby(level='QA年份').mean()
        leads_gain=g.groupby(level='提前期_天').mean()
        win=(g.groupby(level=['QA年份','种子']).mean()>0).mean()
        pass_gate=bool(g.mean()>=R65_MIN_GAIN_KM and (years_gain>0).all() and
                       (leads_gain>=0).all() and win>=2/3 and gn.mean()>0)
        decisions.append({'candidate':name,'gain_vs_LSTM_km':float(g.mean()),
            'gain_vs_GN8_km':float(gn.mean()),'year_gain_km':{str(k):float(v) for k,v in years_gain.items()},
            'lead_gain_km':{str(k):float(v) for k,v in leads_gain.items()},
            'year_seed_winrate':float(win),'qualified':pass_gate})
    wins=[d for d in decisions if d['qualified']]
    selected=max(wins,key=lambda x:x['gain_vs_LSTM_km'])['candidate'] if wins else None
    verdict={'stage':stage,'years':list(years),'selected':selected,'qualified':selected is not None,
             'decisions':decisions,'not_blind':True}
    (REP/f'R65_{stage}_资格判定.json').write_text(json.dumps(verdict,ensure_ascii=False,indent=2),encoding='utf-8')
    display(agg.pivot(index='模型',columns='提前期_天',values='RMSE_km'))
    display(pd.DataFrame(decisions))
    print('[R65 GATE]',stage,'pass',verdict['qualified'],'selected',selected)
    fig,ax=plt.subplots(figsize=(9,5.5),dpi=150)
    for name,g in agg.groupby('模型'):
        ax.plot(g['提前期_天'],g['RMSE_km'],marker='o',label=name)
    ax.set_xlabel('Forecast lead (days)');ax.set_ylabel('Cross-track RMSE (km)')
    ax.set_title(f'R6.5 {stage}: Development errors (not blind)')
    ax.set_xticks(cfg.LEADS);ax.grid(alpha=.2);ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FEN/f'R65_{stage}_TrueRMSE.png',dpi=300)
    fig.savefig(FEN/f'R65_{stage}_TrueRMSE.pdf')
    plt.close(fig)
    return verdict


def r65_run_stage(stage,years):
    try:
        for year in years:
            f=R65_FOLDS[int(year)]
            fd=make_fold_data(f)
            try:
                for seed in R65_SEEDS:
                    for name in R65_MODELS:
                        r65_eval(f,fd,seed,name,stage)
                    r65_archive(f'{stage}_{year}_seed{seed}',False)
            finally:
                del fd;gc.collect()
                if DEV.type=='cuda':torch.cuda.empty_cache()
        decision=r65_audit(stage,years)
        r65_archive(stage+'_complete',True)
        return decision
    except Exception:
        (REP/f'R65_{stage}_运行异常.log').write_text(traceback.format_exc(),encoding='utf-8')
        r65_archive(stage+'_FAILED',True)
        raise

In [ ]:
# ===== R65.5 第一阶段：只训练两个 CrossDrop 新模型 =====
R65_DEV=r65_run_stage('快速资格',R65_EXPLORATORY_YEARS)

In [ ]:
# ===== R65.6 若第一阶段通过，再执行额外开发年份；否则直接跳过 =====
R65_CONFIRM={'qualified':False,'selected':None,'reason':'R65 first-stage gate failed'}
if R65_DEV['qualified']:
    R65_CONFIRM=r65_run_stage('额外年份确认',R65_CONFIRM_YEARS)
else:
    print('[R65] First-stage gate failed; confirmation training skipped.')

In [ ]:
# ===== R65.7 最终诊断报告、完整ZIP和Jupyter下载入口 =====
summary={'version':'R65_exchange_dropout_v1','dataset_sha256':sign,
         'exploratory':R65_DEV,'confirmation':R65_CONFIRM,
         'research_takeaway':'cross-branch coupling regularization vs GN8/LSTMPhys',
         'all_years_not_independent_blind':True,
         'promote_to_full':bool(R65_DEV.get('qualified') and R65_CONFIRM.get('qualified')),
         'completed_at_utc':datetime.now(timezone.utc).isoformat()}
(REP/'R65_最终研究交接.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
r65_archive('final_complete',True)
print('R65 FINAL ZIP:',R65_ARCHIVE)
print('R65 SHA256:',R65_SHA)